# Анализ настроений с отзывами о гостиницах - обработка данных

**ML6 · NLP Fundamentals** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~35 min

## Введение

В этом разделе вы будете использовать техники из предыдущих уроков для проведения разведочного анализа данных большого набора данных. После того как вы получите хорошее представление о полезности различных столбцов, вы узнаете:

- как удалить ненужные столбцы
- как вычислить новые данные на основе существующих столбцов
- как сохранить полученный набор данных для использования в финальном задании


### Введение

До сих пор вы узнали, что текстовые данные сильно отличаются от числовых типов данных. Если это текст, написанный или произнесенный человеком, его можно анализировать, чтобы выявить закономерности, частоты, настроения и смысл. Этот урок познакомит вас с реальным набором данных и реальной задачей: **[515K Hotel Reviews Data in Europe](https://www.kaggle.com/jiashenliu/515k-hotel-reviews-data-in-europe)**, который включает [лицензию CC0: Public Domain](https://creativecommons.org/publicdomain/zero/1.0/). Данные были собраны с Booking.com из открытых источников. Создателем набора данных является Jiashen Liu.

### Подготовка

Вам потребуется:

* Возможность запускать .ipynb ноутбуки с использованием Python 3
* pandas
* NLTK, [который вы должны установить локально](https://www.nltk.org/install.html)
* Набор данных, доступный на Kaggle [515K Hotel Reviews Data in Europe](https://www.kaggle.com/jiashenliu/515k-hotel-reviews-data-in-europe). Его размер составляет около 230 МБ в распакованном виде. Скачайте его в корневую папку `/data`, связанную с этими уроками по NLP.



## Разведочный анализ данных

В этом задании предполагается, что вы создаете бота для рекомендаций гостиниц, используя анализ настроений и оценки гостей. Набор данных, который вы будете использовать, включает отзывы о 1493 различных гостиницах в 6 городах.

Используя Python, набор данных отзывов о гостиницах и анализ настроений NLTK, вы можете выяснить:

* Какие слова и фразы наиболее часто используются в отзывах?
* Соответствуют ли официальные *теги*, описывающие гостиницу, оценкам отзывов (например, больше ли негативных отзывов для определенной гостиницы от *Семей с маленькими детьми*, чем от *Одиноких путешественников*, возможно, указывая на то, что гостиница лучше подходит для *Одиноких путешественников*)?
* Совпадают ли оценки настроений NLTK с числовыми оценками рецензентов?

#### Набор данных

Давайте изучим набор данных, который вы скачали и сохранили локально. Откройте файл в редакторе, например, VS Code или даже Excel.

Заголовки в наборе данных следующие:

*Hotel_Address, Additional_Number_of_Scoring, Review_Date, Average_Score, Hotel_Name, Reviewer_Nationality, Negative_Review, Review_Total_Negative_Word_Counts, Total_Number_of_Reviews, Positive_Review, Review_Total_Positive_Word_Counts, Total_Number_of_Reviews_Reviewer_Has_Given, Reviewer_Score, Tags, days_since_review, lat, lng*

Вот они сгруппированы так, чтобы их было легче изучить:  
##### Столбцы гостиниц

* `Hotel_Name`, `Hotel_Address`, `lat` (широта), `lng` (долгота)
  * Используя *lat* и *lng*, вы можете построить карту с помощью Python, показывающую расположение гостиниц (возможно, с цветовой кодировкой для негативных и позитивных отзывов)
  * Hotel_Address не очевидно полезен для нас, и мы, вероятно, заменим его на страну для более удобной сортировки и поиска

**Столбцы мета-отзывов гостиниц**

* `Average_Score`
  * Согласно создателю набора данных, этот столбец представляет собой *Среднюю оценку гостиницы, рассчитанную на основе последнего комментария за последний год*. Это кажется необычным способом расчета оценки, но это данные, собранные с сайта, поэтому мы можем принять их за данность на данный момент.

  ✅ Исходя из других столбцов в этом наборе данных, можете ли вы придумать другой способ расчета средней оценки?

* `Total_Number_of_Reviews`
  * Общее количество отзывов, которые получила эта гостиница - не ясно (без написания кода), относится ли это к отзывам в наборе данных.
* `Additional_Number_of_Scoring`
  * Это означает, что была дана оценка, но рецензент не написал позитивный или негативный отзыв.

**Столбцы отзывов**

- `Reviewer_Score`
  - Это числовое значение с максимумом 1 десятичным знаком между минимальным и максимальным значениями 2.5 и 10
  - Не объясняется, почему минимальная возможная оценка составляет 2.5
- `Negative_Review`
  - Если рецензент ничего не написал, это поле будет содержать "**No Negative**"
  - Обратите внимание, что рецензент может написать позитивный отзыв в столбце Negative review (например, "в этой гостинице нет ничего плохого")
- `Review_Total_Negative_Word_Counts`
  - Более высокое количество негативных слов указывает на более низкую оценку (без проверки настроения)
- `Positive_Review`
  - Если рецензент ничего не написал, это поле будет содержать "**No Positive**"
  - Обратите внимание, что рецензент может написать негативный отзыв в столбце Positive review (например, "в этой гостинице нет ничего хорошего")
- `Review_Total_Positive_Word_Counts`
  - Более высокое количество позитивных слов указывает на более высокую оценку (без проверки настроения)
- `Review_Date` и `days_since_review`
  - Можно применить меру свежести или устаревания к отзыву (старые отзывы могут быть менее точными, чем новые, потому что управление гостиницей изменилось, были проведены ремонты, добавлен бассейн и т.д.)
- `Tags`
  - Это короткие описатели, которые рецензент может выбрать, чтобы описать тип гостя (например, одиночный или семейный), тип номера, продолжительность пребывания и способ подачи отзыва.
  - К сожалению, использование этих тегов проблематично, ознакомьтесь с разделом ниже, который обсуждает их полезность.

**Столбцы рецензентов**

- `Total_Number_of_Reviews_Reviewer_Has_Given`
  - Это может быть фактором в модели рекомендаций, например, если вы сможете определить, что более плодовитые рецензенты с сотнями отзывов чаще были негативными, чем позитивными. Однако рецензент любого конкретного отзыва не идентифицируется уникальным кодом, и поэтому не может быть связан с набором отзывов. Есть 30 рецензентов с 100 или более отзывами, но трудно понять, как это может помочь модели рекомендаций.
- `Reviewer_Nationality`
  - Некоторые могут думать, что определенные национальности более склонны давать позитивные или негативные отзывы из-за национального уклона. Будьте осторожны, включая такие анекдотические представления в свои модели. Это национальные (а иногда и расовые) стереотипы, и каждый рецензент был индивидуумом, который написал отзыв на основе своего опыта. Он мог быть фильтрован через множество линз, таких как их предыдущие пребывания в гостиницах, пройденное расстояние и их личный темперамент. Считать, что их национальность была причиной оценки, трудно оправдать.

##### Примеры

| Средняя оценка | Общее количество отзывов | Оценка рецензента | Негативный <br />отзыв                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | Позитивный отзыв                 | Теги                                                                                      |
| -------------- | ------------------------ | ----------------- | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------- | ----------------------------------------------------------------------------------------- |
| 7.8            | 1945                    | 2.5               | Это сейчас не гостиница, а строительная площадка. Меня терроризировали с раннего утра и весь день неприемлемым строительным шумом, пока я отдыхал после долгой поездки и работал в номере. Люди работали весь день, например, с отбойными молотками в соседних номерах. Я попросил сменить номер, но тихого номера не было. Чтобы сделать ситуацию хуже, с меня взяли больше, чем нужно. Я выехал вечером, так как у меня был ранний рейс, и получил соответствующий счет. На следующий день гостиница сделала еще один списание без моего согласия, превышающее забронированную цену. Это ужасное место. Не наказывайте себя, бронируя здесь. | Ничего. Ужасное место. Держитесь подальше | Деловая поездка                                Пара Стандартный двухместный номер Пребывание 2 ночи |

Как видите, этот гость остался крайне недоволен своим пребыванием в гостинице. У гостиницы хорошая средняя оценка 7.8 и 1945 отзывов, но этот рецензент дал ей 2.5 и написал 115 слов о том, насколько негативным было его пребывание. Если он ничего не написал в столбце Positive_Review, можно предположить, что ничего позитивного не было, но, увы, он написал 7 слов предупреждения. Если мы просто будем считать слова вместо их смысла или настроения, мы можем получить искаженное представление о намерениях рецензента. Странно, что его оценка 2.5 вызывает путаницу, потому что если пребывание в гостинице было настолько плохим, почему вообще давать какие-либо баллы? Исследуя набор данных внимательно, вы увидите, что минимальная возможная оценка составляет 2.5, а не 0. Максимальная возможная оценка - 10.

##### Теги

Как упоминалось выше, на первый взгляд идея использовать `Tags` для категоризации данных кажется разумной. К сожалению, эти теги не стандартизированы, что означает, что в одной гостинице варианты могут быть *Одноместный номер*, *Двухместный номер* и *Двухместный номер с одной кроватью*, а в другой гостинице - *Делюкс одноместный номер*, *Классический номер с кроватью Queen* и *Эксклюзивный номер с кроватью King*. Это могут быть одни и те же вещи, но существует так много вариаций, что выбор становится:

1. Попытаться изменить все термины на единый стандарт, что очень сложно, потому что не ясно, каким будет путь преобразования в каждом случае (например, *Классический одноместный номер* соответствует *Одноместному номеру*, но *Улучшенный номер с кроватью Queen и видом на сад или город* гораздо сложнее сопоставить)

1. Мы можем использовать подход NLP и измерить частоту определенных терминов, таких как *Одинокий*, *Деловой путешественник* или *Семья с маленькими детьми*, как они применяются к каждой гостинице, и включить это в модель рекомендаций.

Теги обычно (но не всегда) представляют собой одно поле, содержащее список из 5-6 значений, разделенных запятыми, соответствующих *Типу поездки*, *Типу гостей*, *Типу номера*, *Количество ночей* и *Типу устройства, с которого был отправлен отзыв*. Однако, поскольку некоторые рецензенты не заполняют каждое поле (они могут оставить одно пустым), значения не всегда находятся в одном порядке.

Например, возьмем *Тип группы*. В этом поле в столбце `Tags` есть 1025 уникальных возможностей, и, к сожалению, только некоторые из них относятся к группе (некоторые относятся к типу номера и т.д.). Если вы отфильтруете только те, которые упоминают семью, результаты содержат много вариантов типа *Семейный номер*. Если вы включите термин *с*, то есть посчитаете значения *Семья с*, результаты будут лучше, с более чем 80,000 из 515,000 результатов, содержащих фразу "Семья с маленькими детьми" или "Семья с взрослыми детьми".

Это означает, что столбец тегов не полностью бесполезен для нас, но потребуется некоторая работа, чтобы сделать его полезным.

##### Средняя оценка гостиницы

Существует ряд странностей или несоответствий в наборе данных, которые я не могу понять, но они иллюстрируются здесь, чтобы вы знали о них при построении своих моделей. Если вы разберетесь, пожалуйста, сообщите нам в разделе обсуждений!

Набор данных имеет следующие столбцы, относящиеся к средней оценке и количеству отзывов:

1. Hotel_Name
2. Additional_Number_of_Scoring
3. Average_Score
4. Total_Number_of_Reviews
5. Reviewer_Score  

Гостиница с наибольшим количеством отзывов в этом наборе данных - *Britannia International Hotel Canary Wharf* с 4789 отзывами из 515,000. Но если мы посмотрим на значение `Total_Number_of_Reviews` для этой гостиницы, оно составляет 9086. Можно предположить, что существует гораздо больше оценок без отзывов, поэтому, возможно, нам следует добавить значение столбца `Additional_Number_of_Scoring`. Это значение составляет 2682, и добавление его к 4789 дает нам 7471, что все еще на 1615 меньше, чем `Total_Number_of_Reviews`.

Если взять столбец `Average_Score`, можно предположить, что это среднее значение отзывов в наборе данных, но описание на Kaggle гласит: "*Средняя оценка гостиницы, рассчитанная на основе последнего комментария за последний год*". Это не кажется полезным, но мы можем рассчитать собственное среднее значение на основе оценок отзывов в наборе данных. Используя ту же гостиницу в качестве примера, средняя оценка гостиницы указана как 7.1, но рассчитанная оценка (средняя оценка рецензента *в* наборе данных) составляет 6.8. Это близко, но не то же самое значение, и мы можем только предположить, что оценки, данные в `Additional_Number_of_Scoring` отзывах, увеличили среднее значение до 7.1. К сожалению, без возможности проверить или доказать это утверждение, трудно использовать или доверять `Average_Score`, `Additional_Number_of_Scoring` и `Total_Number_of_Reviews`, когда они основаны на данных, которых у нас нет.

Чтобы усложнить ситуацию, гостиница с вторым по величине количеством отзывов имеет рассчитанную среднюю оценку 8.12, а `Average_Score` в наборе данных составляет 8.1. Является ли это совпадением или первая гостиница - несоответствие?

На случай, если эти гостиницы могут быть выбросами, и, возможно, большинство значений совпадают (но некоторые не совпадают по какой-то причине), мы напишем короткую программу, чтобы изучить значения в наборе данных и определить правильное использование (или неиспользование) значений.
> 🚨 Важное предупреждение
>
> Работая с этим набором данных, вы будете писать код, который вычисляет что-то из текста, не читая и не анализируя его самостоятельно. Это суть обработки естественного языка (NLP) — интерпретировать смысл или настроение без участия человека. Однако возможно, что вы столкнетесь с некоторыми негативными отзывами. Я настоятельно рекомендую вам не читать их, потому что в этом нет необходимости. Некоторые из них глупые или неуместные негативные отзывы о гостиницах, например: "Погода была плохая" — то, что находится вне контроля гостиницы или, в принципе, кого-либо. Но есть и темная сторона некоторых отзывов. Иногда негативные отзывы бывают расистскими, сексистскими или эйджистскими. Это неприятно, но ожидаемо для набора данных, собранного с публичного сайта. Некоторые авторы оставляют отзывы, которые могут показаться вам неприятными, дискомфортными или расстраивающими. Лучше позволить коду измерить настроение, чем читать их самостоятельно и расстраиваться. Тем не менее, таких отзывов меньшинство, но они все же существуют.



## Упражнение - Исследование данных

### Загрузка данных

Достаточно визуального изучения данных, теперь вы напишете код и получите ответы! В этом разделе используется библиотека pandas. Ваша первая задача — убедиться, что вы можете загрузить и прочитать данные из CSV. Библиотека pandas имеет быстрый загрузчик CSV, а результат помещается в dataframe, как в предыдущих уроках. CSV, который мы загружаем, содержит более полумиллиона строк, но всего 17 столбцов. Pandas предоставляет множество мощных способов взаимодействия с dataframe, включая возможность выполнять операции над каждой строкой.

С этого момента в уроке будут представлены фрагменты кода, объяснения кода и обсуждение того, что означают результаты. Используйте включенный файл _notebook.ipynb_ для написания вашего кода.

Начнем с загрузки файла данных, который вы будете использовать:



In [ ]:
# Load the hotel reviews from CSV
import pandas as pd
import time
# importing time so the start and end time can be used to calculate file loading time
print("Loading data file now, this could take a while depending on file size")
start = time.time()
# df is 'DataFrame' - make sure you downloaded the file to the data folder
df = pd.read_csv('https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/data/Hotel_Reviews.csv')
end = time.time()
print("Loading took " + str(round(end - start, 2)) + " seconds")

Теперь, когда данные загружены, мы можем выполнять с ними операции. Держите этот код в начале вашей программы для следующей части.



## Исследование данных

В данном случае данные уже *очищены*, то есть они готовы к работе и не содержат символов на других языках, которые могли бы вызвать ошибки в алгоритмах, ожидающих только английские символы.

✅ Возможно, вам придется работать с данными, которые требуют предварительной обработки перед применением методов NLP, но не в этот раз. Если бы вам пришлось, как бы вы обработали символы на других языках?

Убедитесь, что после загрузки данных вы можете исследовать их с помощью кода. Очень легко захотеть сосредоточиться на столбцах `Negative_Review` и `Positive_Review`. Они заполнены естественным текстом для обработки вашими алгоритмами NLP. Но подождите! Прежде чем переходить к NLP и анализу настроений, следуйте приведенному ниже коду, чтобы убедиться, что значения, указанные в наборе данных, совпадают с теми, которые вы вычисляете с помощью pandas.



## Операции с dataframe

Первая задача в этом уроке — проверить, верны ли следующие утверждения, написав код, который исследует dataframe (без его изменения).

> Как и во многих задачах программирования, существует несколько способов выполнения этой задачи, но хороший совет — делать это самым простым и легким способом, особенно если это будет легче понять, когда вы вернетесь к этому коду в будущем. С dataframe есть обширный API, который часто позволяет выполнять задачи эффективно.

Рассматривайте следующие вопросы как задачи по написанию кода и попытайтесь ответить на них, не заглядывая в решение.

1. Выведите *форму* dataframe, который вы только что загрузили (форма — это количество строк и столбцов).
2. Рассчитайте частотное распределение национальностей рецензентов:
   1. Сколько уникальных значений есть в столбце `Reviewer_Nationality` и какие они?
   2. Какая национальность рецензентов является самой распространенной в наборе данных (выведите страну и количество отзывов)?
   3. Какие 10 следующих наиболее часто встречающихся национальностей и их частотное распределение?
3. Какой отель был наиболее часто рецензирован для каждой из 10 наиболее распространенных национальностей рецензентов?
4. Сколько отзывов есть на каждый отель (частотное распределение отзывов по отелям) в наборе данных?
5. Хотя в наборе данных есть столбец `Average_Score` для каждого отеля, вы также можете рассчитать средний балл (вычислив среднее значение всех оценок рецензентов в наборе данных для каждого отеля). Добавьте новый столбец в ваш dataframe с заголовком столбца `Calc_Average_Score`, который содержит этот рассчитанный средний балл.
6. Есть ли отели, у которых одинаковые (округленные до 1 десятичного знака) значения `Average_Score` и `Calc_Average_Score`?
   1. Попробуйте написать функцию на Python, которая принимает Series (строку) в качестве аргумента и сравнивает значения, выводя сообщение, если значения не равны. Затем используйте метод `.apply()`, чтобы обработать каждую строку с помощью функции.
7. Рассчитайте и выведите количество строк, где значения столбца `Negative_Review` равны "No Negative".
8. Рассчитайте и выведите количество строк, где значения столбца `Positive_Review` равны "No Positive".
9. Рассчитайте и выведите количество строк, где значения столбца `Positive_Review` равны "No Positive" **и** значения столбца `Negative_Review` равны "No Negative".

### Ответы в коде

1. Выведите *форму* dataframe, который вы только что загрузили (форма — это количество строк и столбцов).

   ```python
   print("The shape of the data (rows, cols) is " + str(df.shape))
   > The shape of the data (rows, cols) is (515738, 17)
   ```

2. Рассчитайте частотное распределение национальностей рецензентов:

   1. Сколько уникальных значений есть в столбце `Reviewer_Nationality` и какие они?
   2. Какая национальность рецензентов является самой распространенной в наборе данных (выведите страну и количество отзывов)?

   ```python
   # value_counts() creates a Series object that has index and values in this case, the country and the frequency they occur in reviewer nationality
   nationality_freq = df["Reviewer_Nationality"].value_counts()
   print("There are " + str(nationality_freq.size) + " different nationalities")
   # print first and last rows of the Series. Change to nationality_freq.to_string() to print all of the data
   print(nationality_freq) 
   
   There are 227 different nationalities
    United Kingdom               245246
    United States of America      35437
    Australia                     21686
    Ireland                       14827
    United Arab Emirates          10235
                                  ...  
    Comoros                           1
    Palau                             1
    Northern Mariana Islands          1
    Cape Verde                        1
    Guinea                            1
   Name: Reviewer_Nationality, Length: 227, dtype: int64
   ```

   3. Какие 10 следующих наиболее часто встречающихся национальностей и их частотное распределение?

      ```python
      print("The highest frequency reviewer nationality is " + str(nationality_freq.index[0]).strip() + " with " + str(nationality_freq[0]) + " reviews.")
      # Notice there is a leading space on the values, strip() removes that for printing
      # What is the top 10 most common nationalities and their frequencies?
      print("The next 10 highest frequency reviewer nationalities are:")
      print(nationality_freq[1:11].to_string())
      
      The highest frequency reviewer nationality is United Kingdom with 245246 reviews.
      The next 10 highest frequency reviewer nationalities are:
       United States of America     35437
       Australia                    21686
       Ireland                      14827
       United Arab Emirates         10235
       Saudi Arabia                  8951
       Netherlands                   8772
       Switzerland                   8678
       Germany                       7941
       Canada                        7894
       France                        7296
      ```

3. Какой отель был наиболее часто рецензирован для каждой из 10 наиболее распространенных национальностей рецензентов?

   ```python
   # What was the most frequently reviewed hotel for the top 10 nationalities
   # Normally with pandas you will avoid an explicit loop, but wanted to show creating a new dataframe using criteria (don't do this with large amounts of data because it could be very slow)
   for nat in nationality_freq[:10].index:
      # First, extract all the rows that match the criteria into a new dataframe
      nat_df = df[df["Reviewer_Nationality"] == nat]   
      # Now get the hotel freq
      freq = nat_df["Hotel_Name"].value_counts()
      print("The most reviewed hotel for " + str(nat).strip() + " was " + str(freq.index[0]) + " with " + str(freq[0]) + " reviews.") 
      
   The most reviewed hotel for United Kingdom was Britannia International Hotel Canary Wharf with 3833 reviews.
   The most reviewed hotel for United States of America was Hotel Esther a with 423 reviews.
   The most reviewed hotel for Australia was Park Plaza Westminster Bridge London with 167 reviews.
   The most reviewed hotel for Ireland was Copthorne Tara Hotel London Kensington with 239 reviews.
   The most reviewed hotel for United Arab Emirates was Millennium Hotel London Knightsbridge with 129 reviews.
   The most reviewed hotel for Saudi Arabia was The Cumberland A Guoman Hotel with 142 reviews.
   The most reviewed hotel for Netherlands was Jaz Amsterdam with 97 reviews.
   The most reviewed hotel for Switzerland was Hotel Da Vinci with 97 reviews.
   The most reviewed hotel for Germany was Hotel Da Vinci with 86 reviews.
   The most reviewed hotel for Canada was St James Court A Taj Hotel London with 61 reviews.
   ```

4. Сколько отзывов есть на каждый отель (частотное распределение отзывов по отелям) в наборе данных?

   ```python
   # First create a new dataframe based on the old one, removing the uneeded columns
   hotel_freq_df = df.drop(["Hotel_Address", "Additional_Number_of_Scoring", "Review_Date", "Average_Score", "Reviewer_Nationality", "Negative_Review", "Review_Total_Negative_Word_Counts", "Positive_Review", "Review_Total_Positive_Word_Counts", "Total_Number_of_Reviews_Reviewer_Has_Given", "Reviewer_Score", "Tags", "days_since_review", "lat", "lng"], axis = 1)
   
   # Group the rows by Hotel_Name, count them and put the result in a new column Total_Reviews_Found
   hotel_freq_df['Total_Reviews_Found'] = hotel_freq_df.groupby('Hotel_Name').transform('count')
   
   # Get rid of all the duplicated rows
   hotel_freq_df = hotel_freq_df.drop_duplicates(subset = ["Hotel_Name"])
   display(hotel_freq_df) 
   ```
   |                 Hotel_Name                 | Total_Number_of_Reviews | Total_Reviews_Found |
   | :----------------------------------------: | :---------------------: | :-----------------: |
   | Britannia International Hotel Canary Wharf |          9086           |        4789         |
   |    Park Plaza Westminster Bridge London    |          12158          |        4169         |
   |   Copthorne Tara Hotel London Kensington   |          7105           |        3578         |
   |                    ...                     |           ...           |         ...         |
   |       Mercure Paris Porte d Orleans        |           110           |         10          |
   |                Hotel Wagner                |           135           |         10          |
   |            Hotel Gallitzinberg             |           173           |          8          |
   
   Вы можете заметить, что результаты *подсчитанные в наборе данных* не совпадают со значением в `Total_Number_of_Reviews`. Неясно, представляет ли это значение в наборе данных общее количество отзывов, которые получил отель, но не все были собраны, или это какой-то другой расчет. `Total_Number_of_Reviews` не используется в модели из-за этой неясности.

5. Хотя в наборе данных есть столбец `Average_Score` для каждого отеля, вы также можете рассчитать средний балл (вычислив среднее значение всех оценок рецензентов в наборе данных для каждого отеля). Добавьте новый столбец в ваш dataframe с заголовком столбца `Calc_Average_Score`, который содержит этот рассчитанный средний балл. Выведите столбцы `Hotel_Name`, `Average_Score` и `Calc_Average_Score`.

   ```python
   # define a function that takes a row and performs some calculation with it
   def get_difference_review_avg(row):
     return row["Average_Score"] - row["Calc_Average_Score"]
   
   # 'mean' is mathematical word for 'average'
   df['Calc_Average_Score'] = round(df.groupby('Hotel_Name').Reviewer_Score.transform('mean'), 1)
   
   # Add a new column with the difference between the two average scores
   df["Average_Score_Difference"] = df.apply(get_difference_review_avg, axis = 1)
   
   # Create a df without all the duplicates of Hotel_Name (so only 1 row per hotel)
   review_scores_df = df.drop_duplicates(subset = ["Hotel_Name"])
   
   # Sort the dataframe to find the lowest and highest average score difference
   review_scores_df = review_scores_df.sort_values(by=["Average_Score_Difference"])
   
   display(review_scores_df[["Average_Score_Difference", "Average_Score", "Calc_Average_Score", "Hotel_Name"]])
   ```

   Вы также можете задаться вопросом о значении `Average_Score` и почему оно иногда отличается от рассчитанного среднего балла. Поскольку мы не можем знать, почему некоторые значения совпадают, а другие имеют различия, в данном случае безопаснее использовать оценки отзывов, которые у нас есть, чтобы рассчитать среднее значение самостоятельно. Тем не менее, различия обычно очень малы, вот отели с наибольшим отклонением от среднего значения набора данных и рассчитанного среднего:

   | Average_Score_Difference | Average_Score | Calc_Average_Score |                                  Hotel_Name |
   | :----------------------: | :-----------: | :----------------: | ------------------------------------------: |
   |           -0.8           |      7.7      |        8.5         |                  Best Western Hotel Astoria |
   |           -0.7           |      8.8      |        9.5         | Hotel Stendhal Place Vend me Paris MGallery |
   |           -0.7           |      7.5      |        8.2         |               Mercure Paris Porte d Orleans |
   |           -0.7           |      7.9      |        8.6         |             Renaissance Paris Vendome Hotel |
   |           -0.5           |      7.0      |        7.5         |                         Hotel Royal Elys es |
   |           ...            |      ...      |        ...         |                                         ... |
   |           0.7            |      7.5      |        6.8         |     Mercure Paris Op ra Faubourg Montmartre |
   |           0.8            |      7.1      |        6.3         |      Holiday Inn Paris Montparnasse Pasteur |
   |           0.9            |      6.8      |        5.9         |                               Villa Eugenie |
   |           0.9            |      8.6      |        7.7         |   MARQUIS Faubourg St Honor Relais Ch teaux |
   |           1.3            |      7.2      |        5.9         |                          Kube Hotel Ice Bar |

   Поскольку только 1 отель имеет разницу в оценке больше 1, это означает, что мы, вероятно, можем игнорировать разницу и использовать рассчитанный средний балл.

6. Рассчитайте и выведите количество строк, где значения столбца `Negative_Review` равны "No Negative".

7. Рассчитайте и выведите количество строк, где значения столбца `Positive_Review` равны "No Positive".

8. Рассчитайте и выведите количество строк, где значения столбца `Positive_Review` равны "No Positive" **и** значения столбца `Negative_Review` равны "No Negative".

   ```python
   # with lambdas:
   start = time.time()
   no_negative_reviews = df.apply(lambda x: True if x['Negative_Review'] == "No Negative" else False , axis=1)
   print("Number of No Negative reviews: " + str(len(no_negative_reviews[no_negative_reviews == True].index)))
   
   no_positive_reviews = df.apply(lambda x: True if x['Positive_Review'] == "No Positive" else False , axis=1)
   print("Number of No Positive reviews: " + str(len(no_positive_reviews[no_positive_reviews == True].index)))
   
   both_no_reviews = df.apply(lambda x: True if x['Negative_Review'] == "No Negative" and x['Positive_Review'] == "No Positive" else False , axis=1)
   print("Number of both No Negative and No Positive reviews: " + str(len(both_no_reviews[both_no_reviews == True].index)))
   end = time.time()
   print("Lambdas took " + str(round(end - start, 2)) + " seconds")
   
   Number of No Negative reviews: 127890
   Number of No Positive reviews: 35946
   Number of both No Negative and No Positive reviews: 127
   Lambdas took 9.64 seconds
   ```



## Другой способ

Другой способ подсчитать элементы без использования Lambdas — использовать sum для подсчета строк:

   ```python
   # without lambdas (using a mixture of notations to show you can use both)
   start = time.time()
   no_negative_reviews = sum(df.Negative_Review == "No Negative")
   print("Number of No Negative reviews: " + str(no_negative_reviews))
   
   no_positive_reviews = sum(df["Positive_Review"] == "No Positive")
   print("Number of No Positive reviews: " + str(no_positive_reviews))
   
   both_no_reviews = sum((df.Negative_Review == "No Negative") & (df.Positive_Review == "No Positive"))
   print("Number of both No Negative and No Positive reviews: " + str(both_no_reviews))
   
   end = time.time()
   print("Sum took " + str(round(end - start, 2)) + " seconds")
   
   Number of No Negative reviews: 127890
   Number of No Positive reviews: 35946
   Number of both No Negative and No Positive reviews: 127
   Sum took 0.19 seconds
   ```

   Вы могли заметить, что есть 127 строк, где значения столбцов `Negative_Review` и `Positive_Review` равны "No Negative" и "No Positive" соответственно. Это означает, что рецензент дал отелю числовую оценку, но отказался писать как положительный, так и отрицательный отзыв. К счастью, это небольшое количество строк (127 из 515738, или 0.02%), поэтому это, вероятно, не повлияет на нашу модель или результаты в каком-либо конкретном направлении, но вы могли не ожидать, что набор данных отзывов будет содержать строки без отзывов, поэтому стоит исследовать данные, чтобы обнаружить такие строки.

Теперь, когда вы изучили набор данных, в следующем уроке вы будете фильтровать данные и добавлять анализ настроений.



## Задание

Этот урок демонстрирует, как мы видели в предыдущих уроках, насколько важно понимать ваши данные и их особенности перед выполнением операций с ними. Текстовые данные, в частности, требуют тщательного изучения. Исследуйте различные наборы данных с большим количеством текста и попробуйте обнаружить области, которые могут вводить предвзятость или искаженные настроения в модель.



## Обзор и самостоятельное изучение

Пройдите [этот учебный путь по NLP](https://learn.microsoft.com/learn/paths/explore-natural-language-processing/?WT.mc_id=academic-77952-leestott), чтобы узнать о инструментах, которые можно попробовать при создании моделей для работы с речью и текстом.



## Задание

[NLTK](https://github.com/microsoft/ML-For-Beginners/blob/main/6-NLP/4-Hotel-Reviews-1/assignment.md)



---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/6-NLP/4-Hotel-Reviews-1)
- [Учебник ML (ШАД): вероятностный подход](https://contest.yandex.ru/tracks/ml/probabilistic-models/probabilistic-approach-to-machine-learning)
- [Математика для анализа данных: теория вероятностей](https://contest.yandex.ru/tracks/math/probability-theory/chapter-overview)